In [1]:
%load_ext dotenv
%dotenv

In [2]:
%reload_ext dotenv

In [3]:
from llama_index.core import VectorStoreIndex, Settings, Document, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.supabase import SupabaseVectorStore
from datasets import load_dataset
import os
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
SUPABASE_POSTGRES_URI = os.getenv("SUPABASE_POSTGRES_URI")
SUPABASE_COLLECTION = os.getenv("SUPABASE_COLLECTION")
SUPABASE_DIMENSION = int(os.getenv("SUPABASE_DIMENSION"))
SUPABASE_REBUILD = os.getenv("SUPABASE_REBUILD").lower() in {"1", "true", "yes"}

if not SUPABASE_POSTGRES_URI:
    raise ValueError("SUPABASE_POSTGRES_URI environment variable is required.")

In [5]:
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    device="mps",
    embed_batch_size=10,
)

Settings.embed_model = embed_model
Settings.chunk_size = 512
Settings.chunk_overlap = 50

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-m3


In [6]:
def rebuild_index(vector_store: SupabaseVectorStore):
    try:
        news_dataset = load_dataset(
            "RealTimeData/bbc_news_alltime", "2017-12", split="train"
        )
        logging.info(f"Loaded the BBC News dataset with {len(news_dataset)} rows")
        logging.info(f"Successfully loaded the BBC News dataset with {len(news_dataset)} rows.")
    except Exception as e:
        raise ValueError(f"Error loading the BBC News dataset: {str(e)}")

    news_articles = news_dataset["content"]
    unique_articles = set()
    for article in news_articles:
        if article:
            unique_articles.add(article)
    unique_news_articles = list(unique_articles)
    logging.info(f"We have {len(unique_news_articles)} unique articles in our database.")

    articles = [article for article in unique_news_articles if article and len(article) <= 50000]

    documents = [Document(text=t) for t in articles]
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True,
    )
    logging.info("Indexed %s documents in Supabase collection '%s'", len(documents), os.getenv("SUPABASE_COLLECTION"))
    return index

In [7]:
vector_store = SupabaseVectorStore(
    postgres_connection_string=SUPABASE_POSTGRES_URI,
    collection_name=SUPABASE_COLLECTION,
    dimension=SUPABASE_DIMENSION,
)

try:
    if SUPABASE_REBUILD:
        raise RuntimeError("Forced rebuild requested via SUPABASE_REBUILD")
    index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)
    logging.info("Loaded existing Supabase collection '%s'", SUPABASE_COLLECTION)
except Exception as exc:
    logging.info("Creating or refreshing Supabase collection '%s' (%s)", SUPABASE_COLLECTION, exc)
    index = rebuild_index(vector_store)

retriever = index.as_retriever(
    similarity_top_k=5,
)

INFO:root:Loaded existing Supabase collection 'bbc_news'


In [8]:
print(SUPABASE_COLLECTION)

bbc_news


In [9]:
response = retriever.retrieve("Fed rate")
for r in response:
    print("\n----------\n")
    print(r)

/Users/sim/repo/cohort5/.venv/lib/python3.12/site-packages/vecs/collection.py:506: UserWarning: Query does not have a covering index for IndexMeasure.cosine_distance. See Collection.create_index
  warnings.warn(



----------

Node ID: 0bca53e8-bc38-4d05-9fce-4fed1eee3345
Text: Federal Reserve Chairman Janet Yellen has raised interest rates
three times this year  The US Federal Reserve has raised interest
rates by 0.25%, the third rate rise in 2017.  The US central bank said
the move, which was widely expected, underscores "solid" gains in the
US economy.  Officials also boosted their economic forecasts,
projecting 2.5...
Score:  0.299


----------

Node ID: d89bafef-8fc6-4ea4-93e6-3c5b00c1693e
Text: Janet Yellen said that reflected a view in the committee that
the reforms would stimulate consumer spending and business investment.
But there has not been much change in what the Fed's policy makers
think of the longer term prospects. The Fed publishes information
showing the range of expectations that its policy makers have. The
middle of tha...
Score:  0.354


----------

Node ID: e563eb41-fcfa-4645-8b99-fbd0cbcf1024
Text: It attributed the 0.25% rise to record-low unemployment, rising
inflation 